In [ ]:
from pathlib import Path
from dotenv import dotenv_values
from langchain_openai import ChatOpenAI

# 현재 위치와 상위 폴더에서 기존 .env 찾기
current_dir = Path.cwd()
api_key = None

for folder in [current_dir, *current_dir.parents]:
    candidates = [
        folder / ".env",
        folder / "chap08" / ".env",
        folder / "chap07" / ".env",
    ]

    for env_path in candidates:
        if env_path.is_file():
            settings = dotenv_values(env_path)
            key = settings.get("OPENAI_API_KEY")

            if key and key.strip():
                api_key = key.strip()
                print("API 키를 불러온 파일:", env_path)
                break

    if api_key:
        break

# 키를 찾지 못하면 중단
if not api_key:
    raise ValueError(
        "OPENAI_API_KEY를 찾지 못했습니다. 기존 .env 위치를 확인해주세요."
    )

# 랭체인에서 사용할 모델 준비
model = ChatOpenAI(
    model="gpt-4o",
    api_key=api_key,
    temperature=0.7,
    timeout=30,
    max_retries=1,
)

print("랭체인 모델 준비 완료!")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from IPython.display import display, Markdown

# 1. 질문 양식 만들기
# 중괄호 안의 값은 실행할 때 채워진다.
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "너는 {subject} 선생님이야. "
        "대학생이 이해하기 쉽게 비유를 사용해서 설명해줘. "
        "세 문장으로 답하고, 각 문장 사이에 빈 줄을 넣어줘."
    ),
    ("human", "{question}"),
])

# 2. AI 응답에서 텍스트를 꺼내는 파서
parser = StrOutputParser()

# 3. 처리 순서대로 연결
chain = prompt | model | parser

print("LCEL 체인 준비 완료!")

In [ ]:
answer = chain.invoke({
    "subject": "임베디드 시스템",
    "question": "인터럽트가 무엇인지 설명해줘.",
})

display(Markdown(answer))

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
from langchain_core.tools import tool

@tool
def get_current_time(timezone: str) -> str:
    """지정한 IANA 시간대의 현재 시간을 조회한다.
    예: 서울은 Asia/Seoul, 뉴욕은 America/New_York.
    """
    try:
        now = datetime.now(ZoneInfo(timezone))
        return now.strftime("%Y-%m-%d %H:%M:%S %Z")
    except Exception:
        return f"시간 조회 실패: 시간대를 확인해주세요. 입력값: {timezone}"

# 모델에게 사용할 수 있는 도구 알려주기
tools = [get_current_time]
llm_with_tools = model.bind_tools(tools)

print("시간 조회 도구 연결 완료!")

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

tool_messages = [
    SystemMessage(
        content="현재 시간을 물어보면 반드시 시간 조회 도구를 사용해줘."
    ),
    HumanMessage(content="서울은 지금 몇 시야?"),
]

response = llm_with_tools.invoke(tool_messages)

# 도구 호출 요청을 포함한 AI 응답 저장
tool_messages.append(response)

# AI가 어떤 도구를 요청했는지 확인
print(response.tool_calls)

In [ ]:
from langchain_core.messages import ToolMessage

# 도구 이름으로 실제 도구를 찾기 위한 사전
tools_by_name = {t.name: t for t in tools}

for tool_call in response.tool_calls:
    selected_tool = tools_by_name[tool_call["name"]]

    # 실제 파이썬 도구 실행
    result = selected_tool.invoke(tool_call["args"])

    print("도구 실행 결과:", result)

    # 어떤 호출에 대한 결과인지 표시해서 기록
    tool_messages.append(
        ToolMessage(
            content=str(result),
            tool_call_id=tool_call["id"],
        )
    )

In [ ]:
from IPython.display import display, Markdown

# 도구 실행 결과를 포함한 대화를 모델에 전달
final_response = model.invoke(tool_messages)

tool_messages.append(final_response)

display(Markdown(final_response.content))

In [ ]:
import sys
from pathlib import Path

# 현재 위치 또는 상위 폴더에서 chap07 찾기
chap07_dir = None

for folder in [Path.cwd(), *Path.cwd().parents]:
    candidate = folder / "chap07"

    if (candidate / "gpt_functions.py").is_file():
        chap07_dir = candidate
        break

if chap07_dir is None:
    raise FileNotFoundError(
        "chap07/gpt_functions.py를 찾지 못했습니다. 폴더 위치를 확인해주세요."
    )

# 파이썬이 chap07의 모듈을 불러올 수 있도록 경로 추가
if str(chap07_dir) not in sys.path:
    sys.path.insert(0, str(chap07_dir))

# 랭체인 도구와 구분하기 위해 별명 지정
from gpt_functions import get_yf_stock_history as original_stock_history

print("기존 주가 조회 함수 연결 완료!")

In [ ]:
import json
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.tools import tool

# AI가 함수에 전달할 입력 양식
class StockHistoryInput(BaseModel):
    ticker: str = Field(
        description="주식 종목 코드. 예: TSLA, AAPL, MSFT",
        min_length=1,
    )

    period: Literal["5d", "1mo", "3mo", "6mo", "1y"] = Field(
        default="1mo",
        description="조회 기간. 5d, 1mo, 3mo, 6mo, 1y 중 선택",
    )

# 기존 함수를 실행하는 랭체인 도구
@tool(args_schema=StockHistoryInput)
def get_yf_stock_history(
    ticker: str,
    period: str = "1mo",
) -> str:
    """지정한 주식 종목의 일별 주가 기록을 조회한다."""
    result = original_stock_history(
        ticker=ticker,
        period=period,
    )

    return json.dumps(result, ensure_ascii=False, default=str)

# 앞서 만든 시간 도구와 주가 도구를 함께 등록
tools = [get_current_time, get_yf_stock_history]
tools_by_name = {t.name: t for t in tools}

llm_with_tools = model.bind_tools(tools)

print("시간·주가 도구 준비 완료!")

In [ ]:
from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    ToolMessage,
)
from IPython.display import display, Markdown

stock_messages = [
    SystemMessage(
        content=(
            "너는 주가 데이터를 설명하는 도우미야. "
            "주가는 반드시 도구로 조회하고, 조회에 실패하면 추측하지 마. "
            "답변에 실제 데이터의 기간과 통화를 명시해줘. "
            "읽기 쉽게 문단 사이에 빈 줄을 넣어줘."
        )
    ),
    HumanMessage(
        content="테슬라의 최근 5거래일 주가를 조회해서 정리해줘."
    ),
]

# AI가 사용할 도구와 입력값 결정
response = llm_with_tools.invoke(stock_messages)
stock_messages.append(response)

print("AI의 도구 호출 요청:", response.tool_calls)

# 요청한 도구 실행
for tool_call in response.tool_calls:
    selected_tool = tools_by_name[tool_call["name"]]

    try:
        result = selected_tool.invoke(tool_call["args"])
    except Exception as error:
        result = f"도구 실행 실패: {error}"

    stock_messages.append(
        ToolMessage(
            content=str(result),
            tool_call_id=tool_call["id"],
        )
    )

# 조회 결과를 바탕으로 최종 답변 생성
if response.tool_calls:
    final_response = model.invoke(stock_messages)
    stock_messages.append(final_response)
    display(Markdown(final_response.content))
else:
    display(Markdown(response.content))

In [ ]:
from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    ToolMessage,
    AIMessage,
    message_chunk_to_message,
)
from IPython.display import display, Markdown

# 이번 실습용 대화 기록
stream_messages = [
    SystemMessage(
        content=(
            "너는 주가 데이터를 설명하는 도우미야. "
            "주가는 반드시 도구로 조회하고, 실패하면 추측하지 마. "
            "실제 데이터의 기간과 통화를 명시해줘. "
            "문단 사이에는 반드시 빈 줄을 넣어줘."
        )
    ),
    HumanMessage(
        content="테슬라의 최근 5거래일 주가를 조회해서 정리해줘."
    ),
]

# 1. 도구 호출 정보를 스트림으로 받아 모으기
status = display(
    Markdown("사용할 도구를 확인하고 있어요..."),
    display_id=True,
)

gathered = None

for chunk in llm_with_tools.stream(stream_messages):
    if gathered is None:
        gathered = chunk
    else:
        gathered = gathered + chunk

if gathered is None:
    raise RuntimeError("모델 응답을 받지 못했습니다.")

# 조각을 합친 응답을 일반 AI 메시지로 변환
ai_message = message_chunk_to_message(gathered)

if ai_message.invalid_tool_calls:
    raise ValueError("도구 호출 인자가 올바르지 않습니다. 다시 실행해주세요.")

stream_messages.append(ai_message)

# 2. 완성된 호출 정보를 이용해 도구 실행
if ai_message.tool_calls:
    for tool_call in ai_message.tool_calls:
        tool_name = tool_call["name"]
        arguments = tool_call["args"]

        status.update(
            Markdown(f"**`{tool_name}` 도구로 데이터를 조회하고 있어요...**")
        )

        print("실행할 도구:", tool_name)
        print("입력값:", arguments)

        try:
            selected_tool = tools_by_name[tool_name]
            result = selected_tool.invoke(arguments)
        except Exception as error:
            result = f"도구 실행 실패: {error}"

        stream_messages.append(
            ToolMessage(
                content=str(result),
                tool_call_id=tool_call["id"],
            )
        )

    # 3. 조회 결과를 바탕으로 최종 답변 스트림 출력
    status.update(Markdown("조회 결과를 정리하고 있어요..."))
    answer = ""

    for chunk in model.stream(stream_messages):
        if isinstance(chunk.content, str) and chunk.content:
            answer += chunk.content
            status.update(Markdown(answer))

    if not answer.strip():
        raise RuntimeError("최종 답변이 비어 있습니다.")

    # 최종 답변도 대화 기록에 추가
    stream_messages.append(AIMessage(content=answer))

else:
    # 도구 호출 없이 바로 답변한 경우
    status.update(
        Markdown(ai_message.content or "답변 내용이 없습니다.")
    )